# Notebook 1 - Preparacion del modelo con COBRApy

Objetivo:
- Cargar `yeast-GEM.xml` en un objeto `model`.
- Aplicar la preparacion metabolica usada en el pipeline.
- Exportar `out/S.csv`, `out/lb.csv`, `out/ub.csv`.
- Exportar tambien `out/rxn_ids.txt`, `out/met_ids.txt` y `out/model_ready.xml`.


In [1]:
from pathlib import Path
import numpy as np
import cobra
from cobra.util.array import create_stoichiometric_matrix

OUT_DIR = Path("out")
OUT_DIR.mkdir(exist_ok=True)

SBML_PATH = Path("yeast-GEM.xml")
assert SBML_PATH.is_file(), f"No se encontro SBML: {SBML_PATH.resolve()}"

MODEL_READY_SBML = OUT_DIR / "model_ready.xml"
S_FILE = OUT_DIR / "S.csv"
LB_FILE = OUT_DIR / "lb.csv"
UB_FILE = OUT_DIR / "ub.csv"
RXN_FILE = OUT_DIR / "rxn_ids.txt"
MET_FILE = OUT_DIR / "met_ids.txt"

print(f"SBML: {SBML_PATH.resolve()}")
print(f"OUT_DIR: {OUT_DIR.resolve()}")


SBML: C:\Users\ctorrealba\OneDrive - Viña Concha y Toro S.A\Documentos\Doctorado\Artículos\Artículo - Estimación_dFBA\DC_dFBA_Zenteno\yeast-GEM.xml
OUT_DIR: C:\Users\ctorrealba\OneDrive - Viña Concha y Toro S.A\Documentos\Doctorado\Artículos\Artículo - Estimación_dFBA\DC_dFBA_Zenteno\out


In [2]:
model = cobra.io.read_sbml_model(str(SBML_PATH))

print("Modelo cargado")
print(f"- Metabolitos: {len(model.metabolites)}")
print(f"- Reacciones: {len(model.reactions)}")
print(f"- Genes: {len(model.genes)}")

rxn_ids = {r.id for r in model.reactions}
obj_candidates = ["r_2111"]
obj_id = next((rid for rid in obj_candidates if rid in rxn_ids), None)
print(f"- Objetivo candidato: {obj_id}")


Modelo cargado
- Metabolitos: 2806
- Reacciones: 4131
- Genes: 1161
- Objetivo candidato: r_2111


In [3]:
key_rxn_ids = ["r_1761", "r_1714", "r_1709", "r_1992", "r_4046", "r_4598"]
METS_ANAEROBIC_BYPASS = ["s_3714[c]", "s_1198[c]", "s_1203[c]", "s_1207[c]", "s_1212[c]", "s_0529[c]"]

rxn_idx = {r.id: i + 1 for i, r in enumerate(model.reactions)}
met_idx = {m.id: i + 1 for i, m in enumerate(model.metabolites)}

rxn_nms = {r.id: r.name for r in model.reactions}
met_nms = {m.id: m.name for m in model.metabolites}

print()
print("Chequeo de reacciones clave")
for rid in key_rxn_ids:
    found = rid in rxn_idx
    idx = rxn_idx[rid] if found else -1
    name = rxn_nms.get(rid, "Unknown")
    print(f"- {rid:8s} found={found} idx={idx} name={name}")

print()
print("Chequeo de metabolitos clave")
for mid in METS_ANAEROBIC_BYPASS:
    found = mid in met_idx
    alt = mid.replace("[c]", "")
    found_alt = (not found) and (alt in met_idx)
    idx = met_idx[mid] if found else (met_idx[alt] if found_alt else -1)
    print(f"- {mid:10s} found={found or found_alt} idx={idx} name={met_nms.get(mid, met_nms.get(alt, 'Unknown'))}")



Chequeo de reacciones clave
- r_1761   found=True idx=1221 name=ethanol exchange
- r_1714   found=True idx=1180 name=D-glucose exchange
- r_1709   found=True idx=1175 name=D-fructose exchange
- r_1992   found=True idx=1408 name=oxygen exchange
- r_4046   found=True idx=3411 name=non-growth associated maintenance reaction
- r_4598   found=True idx=3948 name=cofactor pseudoreaction

Chequeo de metabolitos clave
- s_3714[c]  found=True idx=2208 name=heme a
- s_1198[c]  found=True idx=942 name=NAD
- s_1203[c]  found=True idx=947 name=NADH
- s_1207[c]  found=True idx=951 name=NADP(+)
- s_1212[c]  found=True idx=955 name=NADPH
- s_0529[c]  found=True idx=400 name=coenzyme A


In [4]:
def find_metabolite(mdl, mid):
    met_ids = {m.id for m in mdl.metabolites}
    if mid in met_ids:
        return mdl.metabolites.get_by_id(mid)
    clean = mid.replace("[c]", "")
    if clean in met_ids:
        return mdl.metabolites.get_by_id(clean)
    return None

print(f"Reacción r_4598 antes: {model.reactions.get_by_id('r_4598').build_reaction_string()}")

removed = 0.0
if "r_4598" in {r.id for r in model.reactions}:
    rcof = model.reactions.get_by_id("r_4598")
    updates = {}
    for mid in METS_ANAEROBIC_BYPASS:
        met = find_metabolite(model, mid)
        if met is None:
            continue
        coef = rcof.metabolites.get(met, 0.0)
        if coef != 0:
            updates[met] = -coef
            removed += abs(coef)
    if updates:
        rcof.add_metabolites(updates)

print(f"- Metabolitos eliminados de r_4598: {len(METS_ANAEROBIC_BYPASS)}")
print(f"Reacción r_4598 actualizada: {model.reactions.get_by_id('r_4598').build_reaction_string()}")

Reacción r_4598 antes: 0.000190000006114133 s_0529 + 9.99999974737875e-06 s_0687 + 1e-06 s_0750 + 0.00264999992214143 s_1198 + 0.000150000007124618 s_1203 + 0.000569999974686652 s_1207 + 0.00270000007003546 s_1212 + 0.000989999971352518 s_1405 + 1.20000004244503e-06 s_1475 + 6.34000025456771e-05 s_1487 + 9.99999997475243e-07 s_3714 --> s_4205
- Metabolitos eliminados de r_4598: 6
Reacción r_4598 actualizada: 9.99999974737875e-06 s_0687 + 1e-06 s_0750 + 0.000989999971352518 s_1405 + 1.20000004244503e-06 s_1475 + 6.34000025456771e-05 s_1487 --> s_4205


In [29]:
# --- Configuracion de bounds con trazabilidad ---

# Index rapido por ID para evitar busquedas repetidas
rxn_by_id = {r.id: r for r in model.reactions}
change_log = []

# Trazas configurables (ajustables)
VITAMIN_TRACE_LB = -1e-2   # traza mas elevada para vitaminas
ION_TRACE_LB = -1e-3       # trazas para iones esenciales


def _fmt_bound(x):
    return f"{x:.3f}" if isinstance(x, (int, float)) else str(x)


def _short(txt, n=70):
    txt = (txt or "").strip()
    return txt if len(txt) <= n else txt[: n - 3] + "..."


def set_bounds_logged(mdl, rid, lb=None, ub=None, group="", note=""):
    rec = {
        "group": group,
        "rid": rid,
        "note": note,
        "status": "MISSING",
        "name": "",
        "old_lb": None,
        "old_ub": None,
        "new_lb": None,
        "new_ub": None,
    }

    if rid not in rxn_by_id:
        print(f"  [MISSING] {rid:8s} | no existe en el modelo")
        change_log.append(rec)
        return rec

    r = rxn_by_id[rid]
    old_lb, old_ub = float(r.lower_bound), float(r.upper_bound)
    new_lb = old_lb if lb is None else float(lb)
    new_ub = old_ub if ub is None else float(ub)

    r.lower_bound = new_lb
    r.upper_bound = new_ub

    rec.update(
        {
            "status": "UPDATED",
            "name": r.name,
            "old_lb": old_lb,
            "old_ub": old_ub,
            "new_lb": new_lb,
            "new_ub": new_ub,
        }
    )
    change_log.append(rec)

    print(
        f"  [OK] {rid:8s} | {_short(r.name, 46):46s} | "
        f"({_fmt_bound(old_lb):>8s}, {_fmt_bound(old_ub):>8s}) -> "
        f"({_fmt_bound(new_lb):>8s}, {_fmt_bound(new_ub):>8s})"
    )
    return rec


def apply_group(group_name, reaction_ids, lb=None, ub=None, note=""):
    print("\n" + "=" * 100)
    print(f"Grupo: {group_name}")
    print(f"Objetivo de bounds: lb={lb if lb is not None else 'sin cambio'}, ub={ub if ub is not None else 'sin cambio'}")
    if note:
        print(f"Nota: {note}")
    print("-" * 100)
    for rid in reaction_ids:
        set_bounds_logged(model, rid, lb=lb, ub=ub, group=group_name, note=note)


# Reglas de manipulacion, agrupadas por sentido biologico/proceso
ALL_N_SOURCE_IDS = [
    "r_1654", "r_1904", "r_1891", "r_1889", "r_1880", "r_1881",
    "r_1873", "r_1879", "r_1810", "r_1906", "r_1911", "r_1914",
    "r_1899", "r_1897", "r_1903", "r_1913", "r_1912", "r_1893",
    "r_1900", "r_1902", "r_1883", "r_1987", "r_1800",
]

KINETIC_N_SOURCE_IDS = [
    "r_1654",  # NH4+
    "r_1879",  # Arg
    "r_1891",  # Gln
    "r_1889",  # Glu
    "r_1906",  # Ser
    "r_1911",  # Thr
    "r_1873",  # Ala
    "r_1912",  # Trp
]

GROUP_RULES = [
    {
        "group": "Anaerobiosis (O2 apagado)",
        "ids": ["r_1992"],
        "lb": 0.0,
        "ub": 0.0,
        "note": "Forzar intercambio de oxigeno en cero.",
    },
    {
        "group": "Suplementos anaerobicos abiertos",
        "ids": ["r_1757", "r_1915", "r_1994", "r_2106", "r_2134", "r_2137", "r_2189"],
        "lb": -1000.0,
        "ub": 0.0,
        "note": "Permitir captacion de suplementos en anaerobiosis.",
    },
    {
        "group": "Shuttles cerrados",
        "ids": ["r_0713", "r_0714", "r_0487"],
        "lb": 0.0,
        "ub": 0.0,
        "note": "Se aplican bounds asimetricos por reaccion.",
    },
    {
        "group": "Fuentes de carbono",
        "ids": ["r_1714", "r_1709"],
        "lb": -1000.0,
        "ub": 0.0,
        "note": "Glucosa y fructosa solo como uptake.",
    },
    {
        "group": "Fuentes de nitrogeno (todos)",
        "ids": ALL_N_SOURCE_IDS,
        "lb": 0.0,
        "ub": 0.0,
        "note": "23 fuentes de N inhabilitadas para captacion.",
    },
    {
        "group": "Fuentes de nitrogeno (medio vinico) - cinetica",
        "ids": KINETIC_N_SOURCE_IDS,
        "lb": -1000.0,
        "ub": 0.0,
        "note": "7 fuentes de N cineticas habilitadas para captacion.",
    },
    {
        "group": "Vitaminas",
        "ids": ["r_2067", "r_1671", "r_1967", "r_2028", "r_1548", "r_2038", "r_1947"],
        "lb": VITAMIN_TRACE_LB,
        "ub": 0.0,
        "note": "Vitaminas en traza elevada (evitar uso masivo como carbono).",
    },
    {
        "group": "Fosfato abierto",
        "ids": ["r_2005"],
        "lb": -1000.0,
        "ub": 0.0,
        "note": "Liberar captacion de fosfato para diagnostico.",
    },
    {
        "group": "Sulfato abierto",
        "ids": ["r_2060"],
        "lb": -1000.0,
        "ub": 0.0,
        "note": "Liberar captacion de sulfato para diagnostico.",
    },
    {
        "group": "Iones esenciales",
        "ids": ["r_2020", "r_1861", "r_4596", "r_4597", "r_4594", "r_4600", "r_4595", "r_4593"],
        "lb": ION_TRACE_LB,
        "ub": 0.0,
        "note": "Iones en traza.",
    },
    {
        "group": "Productos abiertos",
        "ids": ["r_1761", "r_1808", "r_1634", "r_2056", "r_1549", "r_1546", "r_1552", "r_1765", "r_1867", "r_1866", "r_1862", "r_1865"],
        "lb": 0.0,
        "ub": 1000.0,
        "note": "Excrecion permitida.",
    },
    {
        "group": "Agua y protones",
        "ids": ["r_2100", "r_1832"],
        "lb": -1000.0,
        "ub": 1000.0,
    },
]

# Aplicar grupos estandar
for rule in GROUP_RULES:
    # Caso especial: shuttles con limites asimetricos
    if rule["group"] == "Shuttles cerrados":
        print("\n" + "=" * 100)
        print(f"Grupo: {rule['group']}")
        print("Nota: Cierres especificos por reaccion")
        print("-" * 100)
        set_bounds_logged(model, "r_0713", lb=0.0, ub=None, group=rule["group"], note="Cerrar uptake inverso")
        set_bounds_logged(model, "r_0714", lb=0.0, ub=None, group=rule["group"], note="Cerrar uptake inverso")
        set_bounds_logged(model, "r_0487", lb=None, ub=0.0, group=rule["group"], note="Cerrar direccion directa")
        continue

    apply_group(
        rule["group"],
        rule["ids"],
        lb=rule.get("lb", None),
        ub=rule.get("ub", None),
        note=rule.get("note", ""),
    )

# ATP de mantenimiento fijo
print("\n" + "=" * 100)
print("Grupo: ATP de mantenimiento")
print("-" * 100)
set_bounds_logged(model, "r_4046", lb=0.7, ub=0.7, group="ATPM", note="ATPM fijo")

# Resumen global de cambios
updated = [x for x in change_log if x["status"] == "UPDATED"]
missing = [x for x in change_log if x["status"] == "MISSING"]
print("\n" + "=" * 100)
print("Resumen de aplicacion de bounds")
print("=" * 100)
print(f"Total reglas procesadas: {len(change_log)}")
print(f"- Reacciones actualizadas: {len(updated)}")
print(f"- Reacciones faltantes:   {len(missing)}")
if missing:
    print("IDs faltantes:")
    for m in missing:
        print(f"  - {m['rid']} (grupo: {m['group']})")

# Chequeo final de reacciones clave con nombre y ecuacion
KEY_RXN_LABELS = {
    "r_1992": "Intercambio O2",
    "r_1714": "Uptake glucosa",
    "r_1709": "Uptake fructosa",
    "r_4046": "Mantenimiento ATP",
    "r_1761": "Intercambio etanol",
}

print("\n" + "=" * 100)
print("Chequeo final (reacciones clave)")
print("=" * 100)
for rid, label in KEY_RXN_LABELS.items():
    if rid not in rxn_by_id:
        print(f"- {label:20s} | {rid:8s} | MISSING")
        continue
    r = rxn_by_id[rid]
    eq = _short(getattr(r, "reaction", ""), n=90)
    print(
        f"- {label:20s} | {rid:8s} | {_short(r.name, 42):42s} | "
        f"lb={r.lower_bound:.3f}, ub={r.upper_bound:.3f}"
    )
    if eq:
        print(f"    ecuacion: {eq}")



Grupo: Anaerobiosis (O2 apagado)
Objetivo de bounds: lb=0.0, ub=0.0
Nota: Forzar intercambio de oxigeno en cero.
----------------------------------------------------------------------------------------------------
  [OK] r_1992   | oxygen exchange                                | (   0.000,    0.000) -> (   0.000,    0.000)

Grupo: Suplementos anaerobicos abiertos
Objetivo de bounds: lb=-1000.0, ub=0.0
Nota: Permitir captacion de suplementos en anaerobiosis.
----------------------------------------------------------------------------------------------------
  [OK] r_1757   | ergosterol exchange                            | (-1000.000,    0.000) -> (-1000.000,    0.000)
  [OK] r_1915   | lanosterol exchange                            | (-1000.000,    0.000) -> (-1000.000,    0.000)
  [OK] r_1994   | palmitoleate exchange                          | (-1000.000,    0.000) -> (-1000.000,    0.000)
  [OK] r_2106   | zymosterol exchange                            | (-1000.000,    0.000) -> (

In [30]:
# Diagnostico FBA rapido (antes de exportar a Julia)
print("\n" + "=" * 100)
print("Chequeo FBA con bounds actuales")
print("=" * 100)

solution = model.optimize()
print(f"status: {solution.status}")
print(f"objective ({model.objective.expression}): {solution.objective_value:.6g}")

if solution.status != "optimal":
    print("[WARN] El problema FBA no llego a optimo. Revisar bounds y consistencia del modelo.")

# Convencion: biomasa esperada positiva
if solution.objective_value <= 1e-8:
    print("[WARN] Crecimiento nulo o casi nulo (objective <= 1e-8).")
else:
    print("[OK] El modelo muestra crecimiento bajo bounds actuales.")

# Flujos clave: etiqueta biologica + ID + nombre de reaccion
key_flux_map = [
    ("Biomasa", "r_2111"),
    ("Uptake glucosa", "r_1714"),
    ("Uptake fructosa", "r_1709"),
    ("Produccion etanol", "r_1761"),
    ("Intercambio O2", "r_1992"),
    ("Mantenimiento ATP", "r_4046"),
]

print("\nFlujos clave (FBA):")
for label, rid in key_flux_map:
    if rid in rxn_by_id:
        rxn = rxn_by_id[rid]
        v = solution.fluxes.get(rid, float("nan"))
        print(f"- {label:20s} ({rid}) | {rxn.name} | v = {v: .6g}")
    else:
        print(f"- {label:20s} ({rid}) | MISSING")

# Diagnostico adicional de posibles bloqueos (uptakes de C/N)
print("\nUptakes de C/N habilitados (lb, ub, flujo):")
for rid in ["r_1714", "r_1709"] + KINETIC_N_SOURCE_IDS:
    if rid not in rxn_by_id:
        print(f"- {rid:8s}: MISSING")
        continue
    rxn = rxn_by_id[rid]
    v = solution.fluxes.get(rid, float("nan"))
    print(f"- {rxn.name:40s} ({rid}) | lb={rxn.lower_bound:8.3f}, ub={rxn.upper_bound:8.3f}, v={v: .6g}")



Chequeo FBA con bounds actuales
status: optimal
objective (1.0*r_2111 - 1.0*r_2111_reverse_58b69): 0.275482
[OK] El modelo muestra crecimiento bajo bounds actuales.

Flujos clave (FBA):
- Biomasa              (r_2111) | growth | v =  0.275482
- Uptake glucosa       (r_1714) | D-glucose exchange | v = -666.104
- Uptake fructosa      (r_1709) | D-fructose exchange | v = -1000
- Produccion etanol    (r_1761) | ethanol exchange | v =  554.783
- Intercambio O2       (r_1992) | oxygen exchange | v =  0
- Mantenimiento ATP    (r_4046) | non-growth associated maintenance reaction | v =  0.7

Uptakes de C/N habilitados (lb, ub, flujo):
- D-glucose exchange                       (r_1714) | lb=-1000.000, ub=   0.000, v=-666.104
- D-fructose exchange                      (r_1709) | lb=-1000.000, ub=   0.000, v=-1000
- ammonium exchange                        (r_1654) | lb=-1000.000, ub=   0.000, v= 0
- L-arginine exchange                      (r_1879) | lb=-1000.000, ub=   0.000, v= 0
- L-glutami

In [23]:
# FVA dirigido + pruebas de sensibilidad de fuentes de carbono/nitrogeno
print("\n" + "=" * 100)
print("FVA dirigido (diagnostico de degeneracion y alternativas de flujo)")
print("=" * 100)

fva_ids = ["r_2111", "r_1714", "r_1709", "r_1761", "r_1992", "r_4046"] + KINETIC_N_SOURCE_IDS
fva_ids = [rid for rid in fva_ids if rid in rxn_by_id]

fva_df = cobra.flux_analysis.flux_variability_analysis(
    model,
    reaction_list=fva_ids,
    fraction_of_optimum=0.99,
)

print("FVA @99% del optimo (min, max):")
for rid in fva_ids:
    rxn = rxn_by_id[rid]
    mn = float(fva_df.loc[rid, "minimum"])
    mx = float(fva_df.loc[rid, "maximum"])
    print(f"- {rxn.name:40s} ({rid}) | min={mn: .6g}  max={mx: .6g}")

print("\nLectura rapida:")
print("- En exchanges de uptake, valores negativos = consumo posible.")
print("- Si glucosa/fructosa tienen min ~ 0 y aminoacidos min << 0, el modelo prefiere aminoacidos.")


def _solve_with_bounds(base_model, lb_updates):
    m = base_model.copy()
    for rid, new_lb in lb_updates.items():
        if rid in {r.id for r in m.reactions}:
            m.reactions.get_by_id(rid).lower_bound = float(new_lb)
    sol = m.optimize()
    return sol

# Escenario A: bloquear aminoacidos cineticos -> fuerza dependencia de azucares
lb_block_n = {rid: 0.0 for rid in KINETIC_N_SOURCE_IDS if rid in rxn_by_id}
sol_no_n = _solve_with_bounds(model, lb_block_n)

# Escenario B: bloquear azucares -> mide crecimiento posible solo con N/suplementos
lb_block_sugars = {"r_1714": 0.0, "r_1709": 0.0}
sol_no_sugar = _solve_with_bounds(model, lb_block_sugars)

print("\n" + "=" * 100)
print("Pruebas de sensibilidad (objetivo biomasa)")
print("=" * 100)
print(f"- Base (actual):                 status={solution.status:8s}  obj={solution.objective_value: .6g}")
print(f"- Sin aminoacidos cineticos:     status={sol_no_n.status:8s}  obj={sol_no_n.objective_value: .6g}")
print(f"- Sin azucares (glu+fru):        status={sol_no_sugar.status:8s}  obj={sol_no_sugar.objective_value: .6g}")

print("\nInterpretacion:")
print("- Si 'Sin azucares' mantiene crecimiento alto, hay ruta alternativa dominante (ej. aminoacidos).")
print("- Si 'Sin aminoacidos cineticos' cae fuerte o 0, el modelo depende de esas fuentes de N como C/N.")



FVA dirigido (diagnostico de degeneracion y alternativas de flujo)
FVA @99% del optimo (min, max):
- growth                                   (r_2111) | min= 0.0115358  max= 0.0116523
- D-glucose exchange                       (r_1714) | min=-1000  max= 0
- D-fructose exchange                      (r_1709) | min=-1000  max= 0
- ethanol exchange                         (r_1761) | min= 0  max= 1000
- oxygen exchange                          (r_1992) | min= 0  max= 0
- non-growth associated maintenance reaction (r_4046) | min= 0.7  max= 0.7
- ammonium exchange                        (r_1654) | min=-1000  max= 0
- L-arginine exchange                      (r_1879) | min=-1000  max= 0
- L-glutamine exchange                     (r_1891) | min=-1000  max= 0
- L-glutamate exchange                     (r_1889) | min=-1000  max= 0
- L-serine exchange                        (r_1906) | min=-1000  max= 0
- L-threonine exchange                     (r_1911) | min=-1000  max= 0
- L-alanine exchange   

In [31]:
# Aplicar en COBRApy las mismas restricciones clave que en Notebook 2 (aprox. en t=0)
print("\n" + "=" * 100)
print("Imposicion de restricciones estilo Notebook 2 sobre flujos clave")
print("=" * 100)

import re

# Switch principal: sin capeo de productos/biomasa (recomendado para diagnostico FBA/FVA)
APPLY_PRODUCT_CAPS = False
print(f"Modo capeo de producto/biomasa: {'ON' if APPLY_PRODUCT_CAPS else 'OFF (sin capeo)'}")

# --- Parametros (misma base que Notebook 2 por defecto: legacy_calibrated) ---
MU0_nom    = 0.141665
YXN_nom    = 9.80576
YXG_nom    = 0.394345
YXF_nom    = 0.18622
YEG_nom    = 0.14133
YEF_nom    = 0.96932
Kn0_nom    = 0.226882
Kg0_nom    = 3.1514
Kf0_nom    = 2.97625
Kig0_nom   = 29.5276
Kie0_nom   = 2.99809
betaG0_nom = 1.41182
betaF0_nom = 8.49482
MRATE_0    = 0.01
R_GAS      = 8.314
EPS        = 1e-9

MW_N   = 0.014007
MW_GLU = 0.180156
MW_FRU = 0.180156
MW_ETH = 0.046070

# Estado inicial usado en Notebook 2
cX, cN, cG, cF, cE, cO2 = 0.5, 0.14, 110.0, 110.0, 0.0, 0.0
T_val = 293.15  # T_BASE

# Factores de temperatura (mismas ecuaciones)
mu_T  = np.exp(59453.0 * (T_val - 300.0) / (300.0 * R_GAS * T_val))
Kg_T  = np.exp(46055.0 * (T_val - 293.15) / (293.15 * R_GAS * T_val))
b_T   = np.exp(11000.0 * (T_val - 296.15) / (296.15 * R_GAS * T_val))
mrate = MRATE_0 * np.exp(37681.0 * (T_val - 293.30) / (293.30 * R_GAS * T_val))

mu = MU0_nom * mu_T * (cN / (cN + Kn0_nom * Kg_T + EPS))
betaG = betaG0_nom * b_T * (cG / (cG + Kg0_nom * Kg_T + EPS)) * (Kie0_nom * Kg_T / (cE + Kie0_nom * Kg_T + EPS))
betaF = betaF0_nom * b_T * (cF / (cF + Kf0_nom * Kg_T + EPS)) * (Kig0_nom * Kg_T / (cG + Kig0_nom * Kg_T + EPS)) * (Kie0_nom * Kg_T / (cE + Kie0_nom * Kg_T + EPS))

vx = mu
vg = mu / YXG_nom + betaG / YEG_nom + mrate * (cG / (cG + cF + EPS))
vf = mu / YXF_nom + betaF / YEF_nom + mrate * (cF / (cG + cF + EPS))
vn = mu / YXN_nom
ve = (betaG + betaF) / MW_ETH

# Composicion de N (misma usada en notebooks)
N_atoms_map = {
    "r_1654": 1.0,
    "r_1879": 4.0,
    "r_1891": 2.0,
    "r_1889": 1.0,
    "r_1906": 1.0,
    "r_1911": 1.0,
    "r_1873": 1.0,
    "r_1912": 2.0,
}
N_profile_map = {
    "r_1654": 0.40,
    "r_1879": 0.20,
    "r_1891": 0.10,
    "r_1889": 0.05,
    "r_1906": 0.05,
    "r_1911": 0.05,
    "r_1873": 0.05,
    "r_1912": 0.10,
}

N_frac_map = {}
for rid in KINETIC_N_SOURCE_IDS:
    n_atoms = N_atoms_map[rid]
    n_profile = N_profile_map[rid]
    N_frac_map[rid] = n_profile / (n_atoms * MW_N)

# Limites fisicos equivalentes a Notebook 2 (en este snapshot)
lim_glu = vg / MW_GLU
lim_fru = vf / MW_FRU
lim_eth = ve
lim_obj = vx
lim_n_map = {rid: vn * N_frac_map[rid] for rid in KINETIC_N_SOURCE_IDS}

print("Limites calculados (snapshot t=0):")
print(f"- lim_glu uptake  : {lim_glu:.6g}")
print(f"- lim_fru uptake  : {lim_fru:.6g}")
print(f"- lim_eth prod    : {lim_eth:.6g}")
print(f"- lim_obj prod    : {lim_obj:.6g}")
print(f"- limN agregado   : {sum(lim_n_map.values()):.6g}")

# Aplicar bounds sobre modelo actual
# Uptakes: -v <= L  -> lb = -L, ub = 0
if "r_1714" in rxn_by_id:
    rxn_by_id["r_1714"].lower_bound = max(rxn_by_id["r_1714"].lower_bound, -lim_glu)
    rxn_by_id["r_1714"].upper_bound = min(rxn_by_id["r_1714"].upper_bound, 0.0)
if "r_1709" in rxn_by_id:
    rxn_by_id["r_1709"].lower_bound = max(rxn_by_id["r_1709"].lower_bound, -lim_fru)
    rxn_by_id["r_1709"].upper_bound = min(rxn_by_id["r_1709"].upper_bound, 0.0)

# N cinetico por fuente: -v_i <= L_i
for rid in KINETIC_N_SOURCE_IDS:
    if rid in rxn_by_id:
        Li = lim_n_map[rid]
        rxn_by_id[rid].lower_bound = max(rxn_by_id[rid].lower_bound, -Li)
        rxn_by_id[rid].upper_bound = min(rxn_by_id[rid].upper_bound, 0.0)

# Productos/biomasa: opcional
if APPLY_PRODUCT_CAPS:
    if "r_1761" in rxn_by_id:
        rxn_by_id["r_1761"].upper_bound = min(rxn_by_id["r_1761"].upper_bound, lim_eth)
    if "r_2111" in rxn_by_id:
        rxn_by_id["r_2111"].upper_bound = min(rxn_by_id["r_2111"].upper_bound, lim_obj)
else:
    # Sin capeo: liberar cualquier cap previo de sesiones anteriores
    if "r_1761" in rxn_by_id:
        rxn_by_id["r_1761"].upper_bound = max(rxn_by_id["r_1761"].upper_bound, 1000.0)
    if "r_2111" in rxn_by_id:
        rxn_by_id["r_2111"].upper_bound = max(rxn_by_id["r_2111"].upper_bound, 1000.0)

# O2 anaerobio: v=0
if "r_1992" in rxn_by_id:
    rxn_by_id["r_1992"].lower_bound = 0.0
    rxn_by_id["r_1992"].upper_bound = 0.0

print("\nBounds aplicados a flujos clave:")
for rid in ["r_2111", "r_1714", "r_1709", "r_1761", "r_1992", "r_4046"] + KINETIC_N_SOURCE_IDS:
    if rid in rxn_by_id:
        r = rxn_by_id[rid]
        print(f"- {r.name:40s} ({rid}) | lb={r.lower_bound: .6g}, ub={r.upper_bound: .6g}")

# Repetir FBA bajo estos bounds
print("\n" + "=" * 100)
print("Repeticion FBA con restricciones estilo Notebook 2")
print("=" * 100)

solution_nb2 = model.optimize()
print(f"status: {solution_nb2.status}")
print(f"objective ({model.objective.expression}): {solution_nb2.objective_value:.6g}")

print("\nFlujos clave (FBA restringido):")
for label, rid in [
    ("Biomasa", "r_2111"),
    ("Uptake glucosa", "r_1714"),
    ("Uptake fructosa", "r_1709"),
    ("Produccion etanol", "r_1761"),
    ("Intercambio O2", "r_1992"),
    ("Mantenimiento ATP", "r_4046"),
]:
    if rid in rxn_by_id:
        rxn = rxn_by_id[rid]
        v = solution_nb2.fluxes.get(rid, float("nan"))
        print(f"- {label:20s} ({rid}) | {rxn.name} | v = {v: .6g}")

# Diagnostico de fuente de carbono en la solucion
print("\n" + "=" * 100)
print("Diagnostico de fuentes de carbono (intercambios con uptake)")
print("=" * 100)

def carbon_atoms_from_formula(formula):
    if not formula:
        return 0
    m = re.search(r"C(?![a-z])(\d*)", formula)
    if not m:
        return 0
    g = m.group(1)
    return 1 if g == "" else int(g)

carbon_uptakes = []
for ex in model.exchanges:
    rid = ex.id
    v = float(solution_nb2.fluxes.get(rid, 0.0))
    if v < -1e-9:
        met = next(iter(ex.metabolites.keys()))
        c_atoms = carbon_atoms_from_formula(getattr(met, "formula", ""))
        c_rate = (-v) * c_atoms
        carbon_uptakes.append((rid, ex.name, met.id, met.formula, v, c_atoms, c_rate))

carbon_uptakes.sort(key=lambda x: x[6], reverse=True)

if not carbon_uptakes:
    print("No hay uptakes detectados en exchanges (v<0).")
else:
    total_c = sum(x[6] for x in carbon_uptakes)
    print(f"Top fuentes por C-atoms*|v| (total={total_c:.6g}):")
    for rid, rname, mid, formula, v, c_atoms, c_rate in carbon_uptakes[:15]:
        print(f"- {rname:36s} ({rid}) | met={mid:12s} formula={str(formula):10s} | v={v: .6g} | C={c_atoms:2d} | C*|v|={c_rate: .6g}")

    v_glu = float(solution_nb2.fluxes.get("r_1714", 0.0)) if "r_1714" in rxn_by_id else 0.0
    v_fru = float(solution_nb2.fluxes.get("r_1709", 0.0)) if "r_1709" in rxn_by_id else 0.0
    print("\nChequeo rapido azucares:")
    print(f"- r_1714 (glucosa): v={v_glu: .6g}")
    print(f"- r_1709 (fructosa): v={v_fru: .6g}")

# Repetir FVA bajo estos bounds
print("\n" + "=" * 100)
print("Repeticion FVA @99% con restricciones estilo Notebook 2")
print("=" * 100)

fva_ids_nb2 = ["r_2111", "r_1714", "r_1709", "r_1761", "r_1992", "r_4046"] + KINETIC_N_SOURCE_IDS
fva_ids_nb2 = [rid for rid in fva_ids_nb2 if rid in rxn_by_id]
fva_nb2_df = cobra.flux_analysis.flux_variability_analysis(
    model,
    reaction_list=fva_ids_nb2,
    fraction_of_optimum=0.99,
)

print("FVA @99% del optimo (restricciones Notebook 2):")
for rid in fva_ids_nb2:
    rxn = rxn_by_id[rid]
    mn = float(fva_nb2_df.loc[rid, "minimum"])
    mx = float(fva_nb2_df.loc[rid, "maximum"])
    print(f"- {rxn.name:40s} ({rid}) | min={mn: .6g}  max={mx: .6g}")



Imposicion de restricciones estilo Notebook 2 sobre flujos clave
Modo capeo de producto/biomasa: OFF (sin capeo)
Limites calculados (snapshot t=0):
- lim_glu uptake  : 51.9595
- lim_fru uptake  : 10.5261
- lim_eth prod    : 64.7558
- lim_obj prod    : 0.030972
- limN agregado   : 0.169123

Bounds aplicados a flujos clave:
- growth                                   (r_2111) | lb= 0, ub= 1000
- D-glucose exchange                       (r_1714) | lb=-51.9595, ub= 0
- D-fructose exchange                      (r_1709) | lb=-10.5261, ub= 0
- ethanol exchange                         (r_1761) | lb= 0, ub= 1000
- oxygen exchange                          (r_1992) | lb= 0, ub= 0
- non-growth associated maintenance reaction (r_4046) | lb= 0.7, ub= 0.7
- ammonium exchange                        (r_1654) | lb=-0.0901992, ub= 0
- L-arginine exchange                      (r_1879) | lb=-0.0112749, ub= 0
- L-glutamine exchange                     (r_1891) | lb=-0.0112749, ub= 0
- L-glutamate exchange  

In [32]:
# Sensibilidad dual (precios sombra y costos reducidos)
print("\n" + "=" * 100)
print("Analisis de sensibilidad dual (COBRA)")
print("=" * 100)

# Reusar solucion vigente; si no existe, optimizar
sol_dual = solution_nb2 if 'solution_nb2' in globals() else model.optimize()
print(f"status: {sol_dual.status}")
print(f"objective: {sol_dual.objective_value:.6g}")

if sol_dual.status != "optimal":
    print("[WARN] Solucion no optima: precios sombra/costos reducidos pueden ser poco confiables.")

# 1) Precios sombra de metabolitos (dual de balance S*v = 0)
shadow = sol_dual.shadow_prices.copy()
shadow = shadow.replace([np.inf, -np.inf], np.nan).dropna()
shadow_nz = shadow[shadow.abs() > 1e-9]
shadow_top = shadow_nz.reindex(shadow_nz.abs().sort_values(ascending=False).index)

print("\nTop metabolitos por |precio sombra| (abs):")
if shadow_top.empty:
    print("- No hay precios sombra significativos sobre el umbral.")
else:
    for mid, sp in shadow_top.head(20).items():
        met = model.metabolites.get_by_id(mid)
        comp = getattr(met, "compartment", "?")
        print(f"- {met.name:42s} ({mid}) [comp={comp}] | shadow={sp: .6g}")

print("\nInterpretacion rapida precio sombra:")
print("- |shadow| alto => metabolito mas 'caro' para sostener el objetivo (cuello potencial).")
print("- shadow < 0 suele indicar que aumentar disponibilidad de ese balance favoreceria el objetivo.")
print("- shadow > 0 suele indicar presion en sentido opuesto del balance.")

# 2) Costos reducidos de reacciones (dual de bounds)
reduced = sol_dual.reduced_costs.copy()
reduced = reduced.replace([np.inf, -np.inf], np.nan).dropna()
reduced_nz = reduced[reduced.abs() > 1e-9]
reduced_top = reduced_nz.reindex(reduced_nz.abs().sort_values(ascending=False).index)

print("\nTop reacciones por |costo reducido| (abs):")
if reduced_top.empty:
    print("- No hay costos reducidos significativos sobre el umbral.")
else:
    for rid, rc in reduced_top.head(20).items():
        rxn = model.reactions.get_by_id(rid)
        v = float(sol_dual.fluxes.get(rid, np.nan))
        at_lb = abs(v - rxn.lower_bound) <= 1e-7
        at_ub = abs(v - rxn.upper_bound) <= 1e-7
        where = "LB" if at_lb else ("UB" if at_ub else "interior")
        print(f"- {rxn.name:42s} ({rid}) | rc={rc: .6g} | v={v: .6g} | activo_en={where}")

# 3) Exchanges activos en cota (suelen ser cuellos de entrada/salida)
print("\nExchanges activos en cota (|v-lb|<tol o |v-ub|<tol):")
active_ex = []
for ex in model.exchanges:
    rid = ex.id
    v = float(sol_dual.fluxes.get(rid, 0.0))
    at_lb = abs(v - ex.lower_bound) <= 1e-7
    at_ub = abs(v - ex.upper_bound) <= 1e-7
    if at_lb or at_ub:
        active_ex.append((rid, ex.name, v, ex.lower_bound, ex.upper_bound, "LB" if at_lb else "UB"))

if not active_ex:
    print("- Ningun exchange detectado en cota activa.")
else:
    active_ex.sort(key=lambda x: abs(x[2]), reverse=True)
    for rid, name, v, lb, ub, side in active_ex[:25]:
        print(f"- {name:38s} ({rid}) | v={v: .6g} | [{lb: .6g}, {ub: .6g}] | cota={side}")

# 4) Semaforo: cruza exchange en cota + |reduced cost|
print("\n" + "=" * 100)
print("Semaforo de cuellos (exchange en cota + |costo reducido|)")
print("=" * 100)

def bottleneck_level(abs_rc):
    if abs_rc >= 10:
        return "ALTO"
    if abs_rc >= 1:
        return "MEDIO"
    return "BAJO"

def is_actionable_exchange(side, lb, ub):
    blocked = abs(lb) <= 1e-12 and abs(ub) <= 1e-12
    if blocked:
        return False, "bloqueada"

    # Uptake cap activo (flujo en LB negativo): candidato biologicamente accionable
    if side == "LB" and lb < -1e-9:
        return True, "uptake_cap"

    # Product/secreted cap activo (flujo en UB acotado): accionable si no esta abierto a 1000
    if side == "UB" and ub > 1e-9 and ub < 999.0:
        return True, "secretion_cap"

    # Casos tipicos no accionables: LB=0 en solo-secrecion, UB=1000 en abierto
    return False, "estructural"

rc_map = reduced.abs().to_dict()
cross_all = []
cross_actionable = []
for rid, name, v, lb, ub, side in active_ex:
    abs_rc = float(rc_map.get(rid, 0.0))
    if abs_rc <= 1e-9:
        continue

    ok, kind = is_actionable_exchange(side, lb, ub)
    row = (abs_rc, rid, name, side, v, lb, ub, bottleneck_level(abs_rc), kind)
    cross_all.append(row)
    if ok:
        cross_actionable.append(row)

if not cross_all:
    print("- No se detectaron exchanges en cota con costo reducido relevante.")
else:
    cross_all.sort(reverse=True, key=lambda x: x[0])
    cross_actionable.sort(reverse=True, key=lambda x: x[0])

    if cross_actionable:
        print("Top 5 cuellos ACCIONABLES:")
        for abs_rc, rid, name, side, v, lb, ub, lvl, kind in cross_actionable[:5]:
            print(f"- [{lvl:5s}] {name:34s} ({rid}) | |rc|={abs_rc: .6g} | cota={side} | v={v: .6g} | [{lb: .6g}, {ub: .6g}] | tipo={kind}")
    else:
        print("- No hay cuellos accionables bajo este filtro (solo estructurales/bloqueados).")

    print("\nTop 5 cuellos GLOBALES (incluye estructurales):")
    for abs_rc, rid, name, side, v, lb, ub, lvl, kind in cross_all[:5]:
        print(f"- [{lvl:5s}] {name:34s} ({rid}) | |rc|={abs_rc: .6g} | cota={side} | v={v: .6g} | [{lb: .6g}, {ub: .6g}] | tipo={kind}")

print("\nSugerencia:")
print("- Prioriza ajustes en cuellos [ALTO] del bloque ACCIONABLE, luego reoptimiza y vuelve a revisar este semaforo.")


Analisis de sensibilidad dual (COBRA)
status: optimal
objective: 0.0354892

Top metabolitos por |precio sombra| (abs):
- P(1),P(4)-bis(5'-guanosyl) tetraphosphate  (s_1283) [comp=c] | shadow=-1.58276
- P1-(5'-adenosyl),P4-(5'-guanosyl) tetraphosphate (s_1284) [comp=c] | shadow=-1.58276
- P(1),P(3)-bis(5'-adenosyl) triphosphate    (s_4317) [comp=c] | shadow=-1.58276
- P(1),P(4)-bis(5'-adenosyl) tetraphosphate  (s_1282) [comp=c] | shadow=-1.58276
- 5,6,7,8-tetrahydrofolyl-L-glutamic acid    (s_0308) [comp=c] | shadow=-1.26621
- C26:0 chain                                (s_3745) [comp=c] | shadow= 1.26345
- 5-formyltetrahydrofolic acid               (s_0321) [comp=m] | shadow=-1.10793
- 5,10-methylenetetrahydrofolate             (s_0307) [comp=m] | shadow=-1.10793
- 5-formyltetrahydrofolic acid               (s_0320) [comp=e] | shadow=-1.10793
- 5-formyltetrahydrofolic acid               (s_0319) [comp=c] | shadow=-1.10793
- 10-formyl-THF                              (s_0120) [comp=c] |

In [34]:
# Auditoria estructural posterior al analisis de sensibilidad
print("\n" + "=" * 100)
print("Auditoria estructural (folato / SAM / azufre)")
print("=" * 100)

# Solucion de referencia
sol_ref = solution_nb2 if "solution_nb2" in globals() else model.optimize()
print(f"status referencia: {sol_ref.status}")
print(f"objetivo referencia: {sol_ref.objective_value:.6g}")

# 1) Verificar que exchanges estructurales sigan en modo secrecion-only
STRUCTURAL_EXCHANGES = {
    "r_1625": "5-formyltetrahydrofolic acid exchange",
    "r_2043": "S-adenosyl-L-methionine exchange",
    "r_1806": "glutathione disulfide exchange",
    "r_1641": "adenosine 3',5'-bismonophosphate exchange",
    "r_4545": "3'-AMP exchange",
}

print("\n[1] Chequeo de exchanges estructurales (esperado: lb >= 0, ub > 0)")
struct_rows = []
for rid, expected_name in STRUCTURAL_EXCHANGES.items():
    if rid not in rxn_by_id:
        print(f"- {rid}: MISSING")
        continue

    r = rxn_by_id[rid]
    v = float(sol_ref.fluxes.get(rid, 0.0))
    lb, ub = float(r.lower_bound), float(r.upper_bound)

    secretion_only_ok = (lb >= -1e-12) and (ub > 0)
    state = "OK" if secretion_only_ok else "REVISAR"
    print(f"- [{state:7s}] {r.name:45s} ({rid}) | v={v: .6g} | [{lb: .6g}, {ub: .6g}]")

    struct_rows.append((rid, r.name, lb, ub, v, secretion_only_ok))

# 2) Auditoria de candidatos de ruta (folato/SAM/azufre)
keywords = [
    "folate", "tetrahydro", "methion", "adenosyl", "glutathione",
    "sulfate", "sulphate", "paps", "phosphoadenosine", "amp", "sam"
]

def _is_candidate_reaction(rxn):
    txt = (rxn.name or "").lower()
    rid = rxn.id.lower()
    return any(k in txt or k in rid for k in keywords)

candidate_rxns = [r for r in model.reactions if _is_candidate_reaction(r)]

print("\n[2] Reacciones candidatas en modulos folato/SAM/azufre")
print(f"- total candidatas detectadas por palabra clave: {len(candidate_rxns)}")

try:
    blocked_result = cobra.flux_analysis.find_blocked_reactions(model, reaction_list=candidate_rxns)
    blocked_ids = {r.id if hasattr(r, "id") else str(r) for r in blocked_result}
except Exception as e:
    blocked_ids = set()
    print(f"[WARN] No se pudo calcular bloqueadas con find_blocked_reactions: {e}")

blocked_rows = []
active_rows = []
for r in candidate_rxns:
    rid = r.id
    v = float(sol_ref.fluxes.get(rid, 0.0))
    is_blocked = rid in blocked_ids
    row = (rid, r.name, v, float(r.lower_bound), float(r.upper_bound))
    if is_blocked:
        blocked_rows.append(row)
    elif abs(v) > 1e-9:
        active_rows.append(row)

print(f"- candidatas bloqueadas: {len(blocked_rows)}")
print(f"- candidatas activas (|v|>1e-9): {len(active_rows)}")

if blocked_rows:
    print("\nTop bloqueadas (hasta 20):")
    for rid, name, v, lb, ub in sorted(blocked_rows, key=lambda x: x[0])[:20]:
        print(f"- {name:46s} ({rid}) | v={v: .3g} | [{lb: .3g}, {ub: .3g}]")

if active_rows:
    print("\nTop activas por |v| (hasta 20):")
    for rid, name, v, lb, ub in sorted(active_rows, key=lambda x: abs(x[2]), reverse=True)[:20]:
        print(f"- {name:46s} ({rid}) | v={v: .6g} | [{lb: .3g}, {ub: .3g}]")

# 3) Resumen para decision de tesis
n_not_ok = sum(0 if ok else 1 for *_rest, ok in struct_rows)
print("\n[3] Resumen")
print(f"- Exchanges estructurales fuera de secrecion-only: {n_not_ok}")
print(f"- Total candidatas bloqueadas en modulos auditados: {len(blocked_rows)}")

print("\nRecomendacion:")
print("- Mantener exchanges estructurales en secrecion-only (sin uptake).")
print("- Tratar bloqueos persistentes como candidatos de gap-filling o curacion de red.")
print("- Si agregas sink/demanda, documentar como hipotesis de modelado y no como medio vinico real.")


Auditoria estructural (folato / SAM / azufre)
status referencia: optimal
objetivo referencia: 0.0354892

[1] Chequeo de exchanges estructurales (esperado: lb >= 0, ub > 0)
- [OK     ] 5-formyltetrahydrofolic acid exchange         (r_1625) | v= 0 | [ 0,  1000]
- [OK     ] S-adenosyl-L-methionine exchange              (r_2043) | v= 0 | [ 0,  1000]
- [OK     ] glutathione disulfide exchange                (r_1806) | v= 0 | [ 0,  1000]
- [OK     ] adenosine 3',5'-bismonophosphate exchange     (r_1641) | v= 0 | [ 0,  1000]
- [OK     ] 3'-AMP exchange                               (r_4545) | v= 0 | [ 0,  1000]

[2] Reacciones candidatas en modulos folato/SAM/azufre
- total candidatas detectadas por palabra clave: 163
- candidatas bloqueadas: 82
- candidatas activas (|v|>1e-9): 35

Top bloqueadas (hasta 20):
- 5-formethyltetrahydrofolate cyclo-ligase       (r_0084) | v= 0 | [ 0,  1e+03]
- 5-methyltetrahydropteroyltriglutamate-homocysteine S-methyltransferase (r_0085) | v= 0 | [ 0,  1e+03]
- 

In [27]:
# Normalizar tipos de bounds para exportacion SBML segura
for r in model.reactions:
    r.lower_bound = float(r.lower_bound)
    r.upper_bound = float(r.upper_bound)

cobra.io.write_sbml_model(model, str(MODEL_READY_SBML))

S = create_stoichiometric_matrix(model, array_type="dense")
lb = np.array([r.lower_bound for r in model.reactions], dtype=float)
ub = np.array([r.upper_bound for r in model.reactions], dtype=float)
rxn_ids = [r.id for r in model.reactions]
met_ids = [m.id for m in model.metabolites]

np.savetxt(S_FILE, S, delimiter=",")
np.savetxt(LB_FILE, lb, delimiter=",")
np.savetxt(UB_FILE, ub, delimiter=",")
newline = chr(10)
RXN_FILE.write_text(newline.join(rxn_ids), encoding="utf-8")
MET_FILE.write_text(newline.join(met_ids), encoding="utf-8")

print("Export completado")
print(f"- model_ready: {MODEL_READY_SBML}")
print(f"- S shape: {S.shape}")
print(f"- lb shape: {lb.shape}")
print(f"- ub shape: {ub.shape}")
print(f"- rxn ids: {len(rxn_ids)}")
print(f"- met ids: {len(met_ids)}")


Export completado
- model_ready: out\model_ready.xml
- S shape: (2806, 4131)
- lb shape: (4131,)
- ub shape: (4131,)
- rxn ids: 4131
- met ids: 2806


In [28]:
S_chk = np.loadtxt(S_FILE, delimiter=",")
lb_chk = np.loadtxt(LB_FILE, delimiter=",")
ub_chk = np.loadtxt(UB_FILE, delimiter=",")

assert S_chk.shape == (len(met_ids), len(rxn_ids))
assert lb_chk.shape[0] == len(rxn_ids)
assert ub_chk.shape[0] == len(rxn_ids)

print("Validacion final OK")
print(f"- S.csv : {S_FILE.resolve()}")
print(f"- lb.csv: {LB_FILE.resolve()}")
print(f"- ub.csv: {UB_FILE.resolve()}")


Validacion final OK
- S.csv : C:\Users\ctorrealba\OneDrive - Viña Concha y Toro S.A\Documentos\Doctorado\Artículos\Artículo - Estimación_dFBA\DC_dFBA_Zenteno\out\S.csv
- lb.csv: C:\Users\ctorrealba\OneDrive - Viña Concha y Toro S.A\Documentos\Doctorado\Artículos\Artículo - Estimación_dFBA\DC_dFBA_Zenteno\out\lb.csv
- ub.csv: C:\Users\ctorrealba\OneDrive - Viña Concha y Toro S.A\Documentos\Doctorado\Artículos\Artículo - Estimación_dFBA\DC_dFBA_Zenteno\out\ub.csv
